In [ ]:
import json
import glob
import os
from pathlib import Path

EVAL_DIR = Path.cwd().parent / "evaluating"

In [ ]:
NORM = 20.5  # reference mean length; combined_distinct_2 gets penalised for shorter outputs


def eval_losses_from_logs(logs_path):
    """Return {epoch_int: eval_loss} pulled from train_history of the logs file."""
    with open(logs_path, "r", encoding="utf-8") as f:
        logs = json.load(f)
    history = logs[0]["train_history"]
    out = {}
    for row in history:
        if "eval_loss" in row and "epoch" in row:
            out[int(round(row["epoch"]))] = row["eval_loss"]
    return out


def mean_count_of_words(texts):
    counts = [len(t.split()) for t in texts if t and t.strip()]
    return sum(counts) / len(counts) if counts else 0.0


def gen_stats_from_qwen(qwen_path):
    """Return {epoch_int: (len_mean, combined_distinct_2)} from qwen_<config>.json."""
    with open(qwen_path, "r", encoding="utf-8") as f:
        d = json.load(f)
    out = {}
    for ep_key, ep in d.get("epochs", {}).items():
        n = int(ep_key.split("_")[-1])
        gens = ep.get("all_generated") or []
        cd2 = (ep.get("metrics") or {}).get("combined_distinct_2")
        out[n] = (mean_count_of_words(gens), cd2)
    return out


def load_metrics_table(metrics_path, logs_path, qwen_path):
    """Return (epochs_list, columns_list, rows_dict) where rows_dict[epoch][col] = value.

    Adjustments to the raw metrics file:
    - eval_loss is lagged by 1 row in metrics; truthful value comes from logs.
    - len_mean (words/comment) and combined_distinct_x_len = cd2 * min(1, len_mean/NORM)
      are appended, computed from all_generated in qwen_<config>.json.
    """
    with open(metrics_path, "r", encoding="utf-8") as f:
        epochs_map = json.load(f)["epochs"]

    eval_overrides = eval_losses_from_logs(logs_path) if os.path.exists(logs_path) else {}
    gen_stats = gen_stats_from_qwen(qwen_path) if os.path.exists(qwen_path) else {}

    epochs_sorted = sorted(epochs_map.values(), key=lambda m: m["epoch"])
    base_cols = [k for k in epochs_sorted[0].keys() if k != "epoch"]
    columns = base_cols + ["len_mean", "combined_distinct_x_len"]

    rows = {}
    for m in epochs_sorted:
        ep = int(round(m["epoch"]))
        row = {c: m.get(c) for c in base_cols}
        row["eval_loss"] = eval_overrides.get(ep, row.get("eval_loss"))

        len_mean, cd2_from_gen = gen_stats.get(ep, (None, None))
        cd2 = row.get("combined_distinct_2")
        if cd2 is None:
            cd2 = cd2_from_gen
        row["len_mean"] = len_mean
        row["combined_distinct_x_len"] = (
            cd2 * min(1.0, len_mean / NORM) if (cd2 is not None and len_mean is not None) else None
        )
        rows[ep] = row

    return [int(round(m["epoch"])) for m in epochs_sorted], columns, rows


def fmt(v):
    if v is None:
        return "--"
    if isinstance(v, float):
        return f"{v:.4f}" if abs(v) < 1000 else f"{v:.2f}"
    return str(v)


def print_table(name, epochs, columns, rows):
    header = ["epoch"] + columns
    table = [header]
    for ep in epochs:
        table.append([str(ep)] + [fmt(rows[ep][c]) for c in columns])

    widths = [max(len(r[i]) for r in table) for i in range(len(header))]
    sep = "-+-".join("-" * w for w in widths)

    print(f"\n=== {name} ===")
    print(" | ".join(h.ljust(w) for h, w in zip(table[0], widths)))
    print(sep)
    for row in table[1:]:
        print(" | ".join(cell.ljust(w) for cell, w in zip(row, widths)))

In [ ]:
metric_files = sorted(glob.glob(str(EVAL_DIR / "qwen_*_metrics.json")))

tables = {}
for mpath in metric_files:
    config = os.path.basename(mpath).replace("_metrics.json", "")
    lpath = str(EVAL_DIR / f"{config}_logs.json")
    qpath = str(EVAL_DIR / f"{config}.json")
    tables[config] = load_metrics_table(mpath, lpath, qpath)

list(tables.keys())

In [ ]:
for name, (epochs, columns, rows) in tables.items():
    print_table(name, epochs, columns, rows)

In [ ]:
# Compact view — headline numbers per model/epoch, plus the length-adjusted score.
HEADLINE_COLS = [
    "train_loss",
    "eval_loss",
    "distinct_1",
    "distinct_2",
    "combined_distinct_2",
    "len_mean",
    "combined_distinct_x_len",
    "ppl_comments",
    "ppl_wiki",
]

for name, (epochs, columns, rows) in tables.items():
    cols = [c for c in HEADLINE_COLS if c in columns]
    print_table(name, epochs, cols, rows)

In [ ]:
# Per-metric cross-model comparison: rows = epoch, columns = model_name.
def cross_model_view(metric):
    names = list(tables.keys())
    all_epochs = sorted({ep for _, (eps, _, _) in tables.items() for ep in eps})

    header = ["epoch"] + names
    table = [header]
    for ep in all_epochs:
        row = [str(ep)]
        for n in names:
            _, _, rows = tables[n]
            row.append(fmt(rows.get(ep, {}).get(metric)))
        table.append(row)

    widths = [max(len(r[i]) for r in table) for i in range(len(header))]
    print(f"\n### {metric} ###")
    print(" | ".join(h.ljust(w) for h, w in zip(table[0], widths)))
    print("-+-".join("-" * w for w in widths))
    for row in table[1:]:
        print(" | ".join(cell.ljust(w) for cell, w in zip(row, widths)))


for metric in [
    "eval_loss",
    "train_loss",
    "ppl_comments",
    "ppl_wiki",
    "distinct_2",
    "len_mean",
    "combined_distinct_x_len",
]:
    cross_model_view(metric)